<a href="https://colab.research.google.com/github/Hasini2706/AgenticAI/blob/main/AgenticAI_Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai python-dotenv ipywidgets

In [ ]:
import os
from openai import OpenAI

from google.colab import userdata

api_key = userdata.get("groq_api_key")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = "openai/gpt-oss-120b"

print("CampusMate AI Connected!")

CampusMate AI Connected!


In [ ]:
COLLEGE_INFO = {

    "attendance_requirement": "75%",

    "library_timing": "9 AM to 6 PM",

    "college_timing": "9 AM to 4 PM",

    "exam_cell": "Administrative Block, Room 102",

    "placement_cell": "Block C, Second Floor",

    "hostel_timing": "6 AM to 9 PM",

    "fee_payment_mode": "Online and Offline",

    "placement_training": "Available for eligible students"

}

print(COLLEGE_INFO)

{'attendance_requirement': '75%', 'library_timing': '9 AM to 6 PM', 'college_timing': '9 AM to 4 PM', 'exam_cell': 'Administrative Block, Room 102', 'placement_cell': 'Block C, Second Floor', 'hostel_timing': '6 AM to 9 PM', 'fee_payment_mode': 'Online and Offline', 'placement_training': 'Available for eligible students'}


In [ ]:
import json

SYSTEM_PROMPT = f"""
ROLE:

You are CampusMate AI, a helpful college assistant.

FACTS:

{json.dumps(COLLEGE_INFO)}

RULES:

1. Answer only using the provided college information.
2. Never invent examination dates or fee amounts.
3. If information is unavailable, ask the student
   to contact the college administration.
4. Be polite and student-friendly.
5. Keep answers short and clear.

FORMAT:

Respond in simple English.
You can also respond in Telugu when requested.
"""

In [ ]:
def classify_intent(message: str) -> str:

    messages = [

        {
            "role": "system",
            "content": """
            Classify student messages into:
            academic, administrative, complaint, placement.

            Return exactly one category.
            """
        },

        {"role": "user",
         "content": "What is the exam syllabus?"},

        {"role": "assistant",
         "content": "academic"},

        {"role": "user",
         "content": "How can I pay my college fees?"},

        {"role": "assistant",
         "content": "administrative"},

        {"role": "user",
         "content": "The classroom projector is not working."},

        {"role": "assistant",
         "content": "complaint"},

        {"role": "user",
         "content": "When will placement training start?"},

        {"role": "assistant",
         "content": "placement"},

        {"role": "user",
         "content": message}

    ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=messages
    )

    return response.choices[0].message.content.strip().lower()


print(classify_intent(
    "I want to know about placement opportunities."
))

placement


In [ ]:
import json

def extract_complaint(message: str) -> dict:

    prompt = """
    Extract student complaint information.

    Return only valid JSON.

    Format:

    {
        "student_name": "string or null",
        "department": "string or null",
        "issue": "string",
        "location": "string or null",
        "priority": "low, medium or high"
    }
    """

    response = client.chat.completions.create(

        model=MODEL_NAME,

        temperature=0,

        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": message}
        ]

    )

    try:

        return json.loads(
            response.choices[0].message.content
        )

    except json.JSONDecodeError:

        return {"error": "Invalid JSON response"}


result = extract_complaint(
    """
    Hi, I am Hasini from AIML department.
    The projector in classroom 113 is not working.
    We have an important presentation tomorrow.
    """
)

print(json.dumps(result, indent=2))

{
  "student_name": "Hasini",
  "department": "AIML",
  "issue": "The projector in classroom 113 is not working.",
  "location": "classroom 113",
  "priority": "high"
}


In [ ]:
import functools
import time

def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        start = time.time()

        print("Function:", func.__name__)

        result = func(*args, **kwargs)

        end = time.time()

        print("Execution time:", end - start)

        return result

    return wrapper


@log_call
def check_attendance(
    attended: int,
    conducted: int
) -> dict:

    if conducted <= 0 or attended < 0 or attended > conducted:
        return {"error": "Invalid attendance values"}

    percentage = (attended / conducted) * 100

    return {

        "attendance": round(percentage, 2),

        "eligible": percentage >= 75

    }


print(check_attendance(80, 100))

Function: check_attendance
Execution time: 0.000110626220703125
{'attendance': 80.0, 'eligible': True}


In [ ]:
class CampusMateAgent:

    def __init__(self, system_prompt, max_turns=5):

        self.max_turns = max_turns

        self.history = [
            {
                "role": "system",
                "content": system_prompt
            }
        ]


    def reset(self):

        self.history = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            }
        ]


    def chat(self, message: str) -> str:

        if not message.strip():
            return "Please enter a message."

        self.history.append({
            "role": "user",
            "content": message
        })

        messages = (
            [self.history[0]]
            + self.history[1:][-(self.max_turns * 2):]
        )

        try:

            response = client.chat.completions.create(

                model=MODEL_NAME,

                messages=messages,

                temperature=0.3

            )

            reply = response.choices[0].message.content

        except Exception as e:

            reply = "Sorry, the assistant is temporarily unavailable."

            print("Error:", e)

        self.history.append({

            "role": "assistant",

            "content": reply

        })

        return reply


agent = CampusMateAgent(SYSTEM_PROMPT)

print("CampusMate Agent Ready!")

CampusMate Agent Ready!


In [ ]:
print(agent.chat(
    "Hi, my name is Hasini. I am from AIML department."
))

print(agent.chat(
    "What is the minimum attendance requirement?"
))

print(agent.chat(
    "Which department am I studying?"
))

Hello Hasini! 👋  
How can I assist you today? Feel free to ask about attendance, timings, exam cell, placement cell, hostel rules, fee payment, or anything else related to college.
The minimum attendance requirement is **75 %**.
You’re studying in the **AIML (Artificial Intelligence & Machine Learning) department**.


In [ ]:
import ipywidgets as widgets

from IPython.display import display, clear_output

chat_output = widgets.Output()

message_input = widgets.Text(
    placeholder="Ask CampusMate anything...",
    layout=widgets.Layout(width="80%")
)

send_button = widgets.Button(
    description="Send",
    button_style="primary"
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning"
)


def send_message(b):

    message = message_input.value.strip()

    if not message:
        return

    message_input.value = ""

    with chat_output:

        print("You:", message)

        reply = agent.chat(message)

        print("CampusMate:", reply)

        print("-" * 50)


def reset_chat(b):

    agent.reset()

    with chat_output:
        clear_output()

    print("Chat reset!")


send_button.on_click(send_message)

reset_button.on_click(reset_chat)

message_input.on_submit(send_message)


display(widgets.HTML(
    "<h2>🎓 CampusMate AI - College Assistant</h2>"
))

display(chat_output)

display(widgets.HBox([
    message_input,
    send_button,
    reset_button
]))

HTML(value='<h2>🎓 CampusMate AI - College Assistant</h2>')

Output()